# Escenario 2 — Generación de Código con Claude Code

**Dominios:** Claude Code Configuration & Workflows · Context Management & Reliability

## Contexto
Tu equipo usa Claude Code para generación de código, refactoring, debugging y documentación. Necesitás configurarlo para que aplique convenciones consistentes en todo el equipo mediante slash commands, CLAUDE.md, y reglas con scope de ruta.

## Lo que vas a practicar
1. Estructura de archivos CLAUDE.md (jerarquía usuario/proyecto/directorio)
2. Reglas path-scoped en `.claude/rules/`
3. Slash commands con alcance de proyecto
4. Skills con `context: fork`
5. Decisión plan mode vs. ejecución directa
6. Invocación de Claude Code en CI/CD con `-p` flag

**Nota:** Este escenario es principalmente de configuración. Los notebooks muestran los archivos que crearías y demuestran las llamadas API equivalentes.

## Parte 1 — Estructura de Archivos de Configuración

### 1.1 CLAUDE.md a nivel de proyecto

In [ ]:
# Contenido que debería tener .claude/CLAUDE.md del proyecto

CLAUDE_MD_PROYECTO = """
# Proyecto: API de E-Commerce

@import .claude/rules/testing.md
@import .claude/rules/api-conventions.md
@import .claude/rules/security.md

## Estándares Universales
- Stack: Python 3.11, FastAPI, PostgreSQL, Redis
- Formato: Black + isort obligatorio
- Type hints en todas las funciones públicas
- Documentar solo el WHY, no el WHAT

## Para revisiones de código automatizadas
- Reportar: bugs, vulnerabilidades de seguridad, violaciones de API contract
- Ignorar: estilo menor, preferencias personales, patrones locales establecidos
- Severidad HIGH: bugs que afectan comportamiento en producción
- Severidad MEDIUM: degradación de confiabilidad bajo carga
- Severidad LOW: mejoras sin impacto en comportamiento actual
"""

print("Contenido de .claude/CLAUDE.md:")
print(CLAUDE_MD_PROYECTO)

In [ ]:
# Regla con path scope para tests
# Este archivo va en .claude/rules/testing.md

TESTING_RULE = """
---
paths:
  - "**/*.test.py"
  - "**/*_test.py"
  - "tests/**/*.py"
---

# Convenciones de Tests

- Framework: pytest con pytest-asyncio para tests async
- Fixtures disponibles en tests/conftest.py — revisar antes de crear nuevas
- Mockear con pytest-mock, NO con unittest.mock directamente
- Estructura: arrange → act → assert con secciones comentadas
- Nombres: test_should_[do_x]_when_[condition_y]
- Cubrir: happy path + 2 casos edge mínimo por función
- NO acceder a la DB real — usar fixtures con datos de prueba
"""

# Regla con path scope para la API
API_RULE = """
---
paths:
  - "src/api/**/*.py"
  - "src/routers/**/*.py"
---

# Convenciones de API

- Respuestas de error: siempre incluir 'error_code', 'message', 'detail'
- Status codes: 200 OK, 201 Created, 400 Bad Request, 401 Unauthorized, 404 Not Found, 422 Validation
- Paginación: limit/offset con defaults limit=20, max=100
- Validación: usar Pydantic models, no validación manual
- NO exponer stack traces en respuestas de producción
"""

print("Regla de testing (.claude/rules/testing.md):")
print(TESTING_RULE)
print("\nRegla de API (.claude/rules/api-conventions.md):")
print(API_RULE)

### 1.2 Slash Command para revisión de código

In [ ]:
# Este archivo va en .claude/commands/review.md
# Disponible para todo el equipo (via VCS)

REVIEW_COMMAND = """
Ejecutá el checklist estándar de revisión de código del equipo.

Para cada archivo en el diff actual:
1. Verificar bugs y vulnerabilidades de seguridad
2. Verificar cumplimiento de API conventions (si aplica al path)
3. Verificar cobertura de tests (si el cambio modifica lógica de negocio)
4. Verificar manejo de errores

Output formato JSON:
{
  "findings": [
    {
      "file": "src/api/users.py",
      "line": 42,
      "severity": "HIGH",
      "category": "security",
      "description": "...",
      "suggested_fix": "..."
    }
  ],
  "summary": "X issues encontrados: Y HIGH, Z MEDIUM"
}
"""

print("Slash command /review (.claude/commands/review.md):")
print(REVIEW_COMMAND)

### 1.3 Skill con context: fork

In [ ]:
# Este archivo va en .claude/skills/analyze-module.md
# context: fork = corre en subagente aislado, no contamina el contexto principal

ANALYZE_SKILL = """
---
name: analyze-module
description: Análisis profundo de arquitectura de un módulo
context: fork
allowed-tools: Read, Grep, Glob
argument-hint: "Ruta del módulo a analizar (ej: src/api/users)"
---

Analizá el módulo especificado en profundidad:

1. Mapeá todas las dependencias (imports externos e internos)
2. Identificá el patrón arquitectónico usado
3. Listá todas las funciones/clases públicas con su propósito
4. Detectá deuda técnica y code smells
5. Identificá riesgos de seguridad

Generá un reporte estructurado con:
- Resumen ejecutivo (3 líneas)
- Dependencias críticas
- Issues encontrados (severidad + descripción)
- Recomendaciones priorizadas
"""

print("Skill analyze-module (.claude/skills/analyze-module.md):")
print(ANALYZE_SKILL)

print("""
¿Por qué context: fork?
→ El análisis de un módulo produce output verbose que llevaría el contexto principal.
→ Al ejecutarse en subagente aislado, solo vuelve el resumen estructurado.
→ El contexto de la sesión principal queda limpio para la tarea real.
""")

## Parte 2 — Decisión Plan Mode vs. Ejecución Directa

In [ ]:
# Tabla de decisión

casos = [
    {
        "tarea": "Arreglar bug: NullPointerException en users.py línea 45",
        "modo": "EJECUCIÓN DIRECTA",
        "razón": "Bug específico, stack trace claro, un solo archivo, alcance delimitado"
    },
    {
        "tarea": "Reestructurar el monolito en microservicios (45+ archivos)",
        "modo": "PLAN MODE",
        "razón": "Cambios a gran escala, múltiples enfoques válidos, decisiones arquitectónicas"
    },
    {
        "tarea": "Migrar de requests a httpx (afecta 15 archivos)",
        "modo": "PLAN MODE",
        "razón": "Multi-archivo, diferencias de API entre librerías, riesgo de regresiones"
    },
    {
        "tarea": "Agregar validación de fecha a un campo de formulario",
        "modo": "EJECUCIÓN DIRECTA",
        "razón": "Cambio simple, bien delimitado, sin dependencias"
    },
    {
        "tarea": "Integrar sistema de caching (Redis vs. Memcached vs. in-memory)",
        "modo": "PLAN MODE",
        "razón": "Múltiples enfoques válidos, implicaciones de infraestructura, decisión arquitectónica"
    }
]

print(f"{'TAREA':<55} {'MODO':<20} RAZÓN")
print("-" * 120)
for caso in casos:
    print(f"{caso['tarea']:<55} {caso['modo']:<20} {caso['razón']}")

## Parte 3 — Claude Code en CI/CD

In [ ]:
# Equivalente API de lo que hace claude -p en CI
import anthropic

client = anthropic.Anthropic()

# Simular un diff de PR
pr_diff = """
diff --git a/src/api/users.py b/src/api/users.py
@@ -40,8 +40,12 @@ class UserRouter:
     async def get_user(self, user_id: str, db: Session = Depends(get_db)):
-        user = db.query(User).filter(User.id == user_id).first()
+        query = f"SELECT * FROM users WHERE id = '{user_id}'"
+        user = db.execute(query).first()
         if not user:
             raise HTTPException(status_code=404)
         return user
"""

# Schema de output estructurado para CI
review_tool = {
    "name": "report_findings",
    "description": "Reporta los hallazgos de la revisión de código",
    "input_schema": {
        "type": "object",
        "properties": {
            "findings": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "file": {"type": "string"},
                        "line": {"type": "integer"},
                        "severity": {"type": "string", "enum": ["HIGH", "MEDIUM", "LOW"]},
                        "category": {"type": "string", "enum": ["security", "bug", "performance", "style"]},
                        "description": {"type": "string"},
                        "suggested_fix": {"type": ["string", "null"]}
                    },
                    "required": ["file", "severity", "category", "description"]
                }
            },
            "summary": {"type": "string"}
        },
        "required": ["findings", "summary"]
    }
}

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    system="""Revisá el diff de PR. 
Reportar SOLO: bugs confirmados, vulnerabilidades de seguridad.
NO reportar: estilo, preferencias personales.
Criterio HIGH: comportamiento incorrecto en producción.
Criterio MEDIUM: degradación bajo carga.""",
    tools=[review_tool],
    tool_choice={"type": "tool", "name": "report_findings"},
    messages=[{"role": "user", "content": f"Revisá este diff:\n\n{pr_diff}"}]
)

import json
for block in response.content:
    if block.type == "tool_use":
        print("Hallazgos de la revisión (output estructurado para CI):")
        print(json.dumps(block.input, indent=2, ensure_ascii=False))

## Preguntas de Reflexión

1. Un nuevo dev clona el repo y no ve las reglas de testing aplicadas. ¿Dónde están probablemente esas reglas?
2. ¿Por qué la skill `analyze-module` usa `context: fork`? ¿Qué pasaría si no lo usara?
3. ¿Cuál es la diferencia entre poner las convenciones de tests en `.claude/rules/testing.md` con glob `**/*.test.py` vs. en un CLAUDE.md en el directorio `tests/`?
4. ¿Qué flag necesitás agregar al comando `claude` para que no se cuelgue en CI/CD?
5. ¿Por qué es mejor que la instancia de CI que revisa el PR sea distinta a la que lo generó?